## Imports

Brings in `matplotlib` for plotting the results, `torch`/`nn`/`AdamW` for training, and the project's own `get_dataloaders`, `Transformer`, and `compute_l2_norm` (the first predictor being benchmarked).

In [ ]:
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.optim import AdamW

from data.modular_arithmetic import get_dataloaders
from models.transformer import Transformer
from predictors.l2_norm import compute_l2_norm

## Selecting compute device (MPS or CPU)

Picks Apple Silicon's `mps` backend if available, otherwise falls back to `cpu` — the project's hardware target per the benchmark protocol. All batches and the model itself are later moved onto this device.

In [ ]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Device:", device)

## Loading the train/test data

Builds the train/test `DataLoader`s for `(a+b) mod 97`, using a **full-batch** size (`0.3 * 97 * 97`, i.e. the entire train split in one batch) — this matches Nanda et al.'s training regime and is required for the aggressive `weight_decay=1.0` used below to behave correctly.

In [ ]:
data_loader = get_dataloaders(97, batch_size=int(0.3 * 97 * 97))

## Instantiating the model

Creates the `Transformer` with the real task's dimensions (`vocab_size=98`, `d_model=128`) and moves it onto the selected device (`mps` or `cpu`).

In [ ]:
model = Transformer(vocab_size=98, d_model=128).to(device)

## Setting up the optimizer

Uses `AdamW` with a strong `weight_decay=1.0` — Nanda et al.'s value, which is what actually drives the model past pure memorization into the generalizing (grokked) solution when paired with full-batch training.

In [ ]:
optimizer = AdamW(model.parameters(), lr=1e-3, weight_decay=1.0)

## Confirming optimizer configuration

Prints the optimizer's settings as a quick manual check that `AdamW` was constructed with the intended learning rate and weight decay.

In [ ]:
print("Optimizer:", optimizer)

## Defining the loss function

Standard multi-class cross-entropy loss, applied later only to the logits at the `"="` position — this is what's being minimized to predict `(a + b) % 97`.

In [ ]:
cross_entropy_loss = nn.CrossEntropyLoss()

## Tracking metrics across training

Sets the number of epochs (`20000`, needed since grokking can take thousands of steps to occur) and initializes the lists used to record train accuracy, test accuracy, loss, and the model's L2 weight norm at every epoch — the data that later gets plotted.

In [ ]:
num_epochs = 20000
train_acc_history = []
test_acc_history = []
loss_history = []
l2_norm_history = []

## The training loop (M1 gate)

For each epoch: runs one full-batch gradient step on the train set (forward pass on the `"="` position's logits, cross-entropy loss, backward pass, optimizer step) while tracking train accuracy, then evaluates on the test set with no gradient updates to track test accuracy. Every 100 epochs it prints progress, and every epoch it records the model's L2 norm via `compute_l2_norm` — the signal the L2 Norm predictor is based on. This is the loop that reproduces the grokking curve (train accuracy rising early, test accuracy lagging, then sharply catching up).

In [ ]:
for epoch in range(num_epochs):
    total_correct = 0
    total_samples = 0
    for x, y in data_loader[0]:
        x, y = x.to(device), y.to(device)
        logit = model.forward(x)
        equal_sign_logit = logit[:, 2, :]

        loss = cross_entropy_loss(equal_sign_logit, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        predicted = equal_sign_logit.argmax(dim=1)
        total_correct += (predicted == y).sum().item()
        total_samples += len(y)
        
    test_total_correct = 0
    test_total_samples = 0
        
    for x_test, y_test in data_loader[1]:
        x_test, y_test = x_test.to(device), y_test.to(device)
        logit_test = model.forward(x_test)
        equal_sign_logit_test = logit_test[:, 2, :]
        predicted_test = equal_sign_logit_test.argmax(dim=1)
        test_total_correct += (predicted_test == y_test).sum().item()
        test_total_samples += len(y_test)

    train_acc_history.append(total_correct / total_samples)
    test_acc_history.append(test_total_correct / test_total_samples)
    loss_history.append(loss.item())
    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch + 1}: Loss = {loss_history[-1]}, Train Acc = {train_acc_history[-1]}, Test Acc = {test_acc_history[-1]}")

    
    l2_norm = compute_l2_norm(model)
    l2_norm_history.append(l2_norm)

## Plotting the grokking curve and L2 norm curve

Plots train vs. test accuracy against epoch on a log-scale x-axis (the grokking transition only shows up as a sharp elbow on a log axis) and saves it as `grokking_curve.png`, then plots the L2 norm history the same way and saves it as `l2_norm_curve.png`.

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(range(1, num_epochs + 1), train_acc_history, label="Train Accuracy")
plt.plot(range(1, num_epochs + 1), test_acc_history, label="Test Accuracy")
plt.xscale("log")
plt.xlabel("Epoch (log scale)")
plt.ylabel("Accuracy")
plt.title("Grokking Curve")
plt.legend()
plt.savefig("grokking_curve.png")
plt.figure(figsize=(8, 5))
plt.plot(range(1, num_epochs + 1), l2_norm_history, label="L2 Norm")
plt.xscale("log")
plt.xlabel("Epoch (log scale)")
plt.ylabel("L2 Norm")
plt.title("Weight Norm Over Training")
plt.legend()
plt.savefig("l2_norm_curve.png")

plt.show()